# RAG Evaluation with Grouse — LLM-as-Judge for Retrieval Systems

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/evaluation_grouse.ipynb)

**A hands-on guide to building rubric-based, LLM-as-judge evaluation pipelines for
Retrieval-Augmented Generation systems.** Unlike metric-based approaches that reduce quality
to a single float, the *Grouse* (Grounded Scoring Engine) paradigm uses a language model
to evaluate RAG outputs against structured rubrics — producing not just scores but
natural-language justifications that explain *why* an answer succeeds or fails.

In this notebook we build the entire judge pipeline from scratch: retrieval, generation,
rubric design, prompt engineering for the judge, score extraction, and aggregation.
Because we use the same model for generation and judging, we also explore the
*circular evaluation* problem and discuss practical mitigation strategies.

| Component | Implementation |
|---|---|
| **LLM** | Qwen/Qwen3-8B (4-bit NF4 quantization) |
| **Embeddings** | BAAI/bge-base-en-v1.5 (768-dim, sentence-transformers) |
| **Vector Store** | FAISS (faiss-cpu) with cosine similarity |
| **Evaluation Approach** | LLM-as-Judge with structured rubrics (Grouse-style) |
| **Data** | Synthetic inline knowledge base (no external files) |

> **Runtime:** Google Colab with T4 GPU. Estimated setup time: ~3 minutes.
> The notebook is self-contained — no API keys or external services required.

## 1.1 — Learning Objectives

By the end of this notebook you will be able to:

1. **Understand the LLM-as-judge paradigm** for RAG evaluation and how it differs
   from automated metrics like BLEU, ROUGE, or embedding similarity.
2. **Build structured evaluation rubrics** for four core dimensions: groundedness,
   completeness, relevance, and coherence.
3. **Implement an LLM-based judge pipeline from scratch** — including prompt
   engineering, score extraction, and result aggregation.
4. **Compare rubric-based evaluation with automated metric approaches** (cross-reference
   with the DeepEval notebook in this repository).
5. **Identify the circular evaluation limitation** when the same model serves as
   both generator and judge, and apply mitigation strategies.
6. **Design custom rubrics** for domain-specific RAG applications (medical, legal, etc.).

## 1.2 — Limitations of Automated Metrics

Automated evaluation metrics have been the workhorse of NLP evaluation for decades.
BLEU measures n-gram overlap with reference translations; ROUGE counts shared
subsequences between generated and reference summaries; BERTScore computes
token-level cosine similarities in embedding space. These metrics are fast,
deterministic, and easy to integrate into CI/CD pipelines.

But they share a fundamental limitation: **they capture statistical patterns,
not meaning.** A factually correct but poorly structured answer might score
well on faithfulness metrics yet be nearly useless to a human reader. Conversely,
a beautifully written answer that introduces a single hallucinated fact could
score perfectly on coherence metrics while being dangerously wrong.

Consider what these metrics *cannot* assess:

- **Coherence of reasoning** — Does the answer follow a logical chain, or does
  it jump between unrelated points?
- **Appropriate level of detail** — Is the answer too terse, or does it include
  irrelevant padding?
- **Tone and register** — Is the answer appropriately professional for the context?
- **Completeness relative to intent** — Did the system actually answer what the
  user *meant* to ask, not just what they literally typed?

The history of evaluation in NLP follows a clear arc: **expensive human evaluation**
→ **cheap automated metrics** → **LLM-as-judge as a middle ground**. The LLM-as-judge
paradigm attempts to combine the nuance of human evaluation with the scalability of
automated approaches. Instead of counting n-grams, we ask a capable language model
to evaluate outputs the way a human reviewer would — following rubrics, providing
justifications, and assigning scores on meaningful scales.

> **Key insight:** Automated metrics tell you *that* something is wrong;
> LLM judges tell you *what* is wrong and *why*. Both have a place in a
> mature evaluation strategy.

## 1.3 — The LLM-as-Judge Paradigm

The core idea behind LLM-as-judge is deceptively simple: instead of computing a
statistical metric, we *prompt* a language model to evaluate an output the way
a trained human annotator would. The model receives the original question, the
retrieved context, the generated answer, and a rubric — then produces a score
and justification.

What makes this powerful is the **rubric**. Unlike automated metrics, which have
fixed definitions baked into their code, LLM judges follow rubrics expressed in
natural language. This means you can customize evaluation criteria without writing
new code — just write a new rubric. Need to evaluate medical safety? Write a rubric.
Legal citation accuracy? Write a rubric. Customer service tone? Write a rubric.

### Advantages

- **Nuanced assessment** — Can evaluate open-ended, subjective qualities that
  resist quantification.
- **Explainable scores** — Every score comes with a natural-language justification.
- **Customizable** — Rubrics can be tuned per domain, use case, or audience.
- **No reference needed** — Unlike BLEU/ROUGE, no gold-standard answer is required.

### Challenges

- **Judge model biases** — The judge may have systematic preferences (e.g., longer
  answers, certain writing styles).
- **Position bias** — When comparing two answers, the judge may favour the one
  presented first.
- **Self-preference bias** — A model tends to rate its own outputs more favourably
  than outputs from other models.
- **Cost** — Each evaluation requires an LLM inference pass, which can be expensive
  at scale.

The table below summarises how the three evaluation paradigms compare:

| Aspect | Automated Metrics | LLM-as-Judge | Human Evaluation |
|---|---|---|---|
| **Cost** | Low | Medium | High |
| **Speed** | Fast | Medium | Slow |
| **Nuance** | Low | High | Very High |
| **Consistency** | Perfect | High | Variable |
| **Customisable** | Limited | Very | Very |
| **Explainability** | None | High | High |
| **Reference needed** | Usually | No | No |

## 1.4 — Grouse Evaluation Dimensions

The Grouse (Grounded Scoring Engine) framework evaluates RAG outputs across
four complementary dimensions. Each dimension captures a distinct failure mode
and is scored on a 0–5 scale with clear, observable criteria at each level.

**Groundedness** asks: *Is every claim in the answer supported by the retrieved
context?* This is the RAG equivalent of factual accuracy — but scoped to the
context window rather than world knowledge. A score of 5 means zero
extrapolation; a score of 0 means the answer is entirely fabricated.

**Completeness** asks: *Does the answer cover all relevant information present
in the context?* A system might be perfectly grounded but still fail by
cherry-picking a single sentence while ignoring three other relevant paragraphs.

**Relevance** asks: *Does the answer focus on what the user actually asked?*
An answer can be grounded and complete yet still miss the mark if it addresses
a different question than the one posed.

**Coherence** asks: *Is the answer well-structured, clear, and logically
organised?* Even if all the right information is present, a disorganised
wall of text degrades user experience.

Together, these four dimensions form a **diagnostic grid**. A system that scores
high on groundedness but low on completeness is being *too conservative* — it
only states what it is certain about, at the cost of omitting useful information.
A system that scores high on completeness but low on coherence is *dumping
everything* without organising it. The grid helps you pinpoint *which* aspect
of the pipeline to improve.

> **Distinction from DeepEval:** DeepEval metrics (faithfulness, answer relevancy)
> produce binary or float scores via decomposition algorithms. Grouse-style
> evaluation produces *rubric-scored assessments with explanations* — trading
> speed for interpretability.

| Dimension | Question It Answers | Failure Mode Detected |
|---|---|---|
| **Groundedness** | Is every claim supported by context? | Hallucination |
| **Completeness** | Does it cover all relevant context? | Information omission |
| **Relevance** | Does it answer what was asked? | Off-topic response |
| **Coherence** | Is it well-structured and clear? | Poor readability |

## 2.1 — Environment Setup

We install the same core stack used throughout this repository: `sentence-transformers`
for embeddings, `faiss-cpu` for vector search, and `transformers` with `bitsandbytes`
for 4-bit quantised inference. No evaluation-specific packages are needed — we build
the judge pipeline entirely from scratch.

In [ ]:
!pip install -q sentence-transformers faiss-cpu \
    transformers torch bitsandbytes accelerate

## 2.2 — Imports

In [ ]:
import torch
import json
import re
import textwrap
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## 2.3 — Knowledge Base

We create a synthetic knowledge base about a fictional company called **Meridian
Technologies**. These 15 document chunks simulate realistic RAG retrieval
scenarios — policies overlap, some are tangentially related, and a few are
deliberately narrow to test completeness.

In [ ]:
KNOWLEDGE_BASE = [
    "Meridian Technologies offers a flexible remote work policy. Full-time employees "
    "may work remotely up to 3 days per week, subject to manager approval. Teams in "
    "client-facing roles may have stricter in-office requirements.",

    "Remote work equipment stipends are available for employees who work from home at "
    "least 2 days per week. The stipend covers up to $500 annually for ergonomic "
    "furniture, monitors, and peripherals. Receipts must be submitted through the "
    "HR portal.",

    "Performance reviews at Meridian Technologies occur semi-annually in June and "
    "December. Each review includes a self-assessment, peer feedback from at least "
    "two colleagues, and a manager evaluation. Ratings follow a 5-point scale from "
    "Needs Improvement to Exceptional.",

    "The performance review rating directly influences annual compensation "
    "adjustments. Employees rated Exceptional receive 8-12% raises, Strong "
    "Performer receive 4-7%, Meets Expectations receive 2-3%, and lower ratings "
    "may trigger a performance improvement plan.",

    "Meridian Technologies provides a professional development budget of $2,000 "
    "per employee per year. This can be used for conferences, online courses, "
    "certifications, and technical books. Approval from a direct manager is "
    "required for expenses over $500.",

    "The company sponsors internal tech talks every Friday afternoon. Employees "
    "who present receive a $100 gift card and priority consideration during "
    "promotion cycles. Topics range from technical deep-dives to soft skills "
    "workshops.",

    "Health insurance at Meridian Technologies includes three plan tiers: Basic, "
    "Standard, and Premium. All full-time employees are eligible from their start "
    "date. The company covers 80% of premiums for the Standard plan.",

    "Dental and vision insurance are included in the Standard and Premium health "
    "plans. The Basic plan covers only medical. Dependents can be added to any "
    "plan tier at the employee's expense.",

    "Meridian Technologies offers 20 days of paid time off (PTO) per year for "
    "employees with less than 5 years of tenure. After 5 years, PTO increases "
    "to 25 days. Unused PTO can be carried over up to 5 days into the next "
    "calendar year.",

    "Parental leave at Meridian includes 16 weeks of paid leave for primary "
    "caregivers and 6 weeks for secondary caregivers. Adoption and surrogacy "
    "are covered under the same policy. Leave can be taken in non-consecutive "
    "blocks within 12 months.",

    "The engineering department uses a quarterly OKR (Objectives and Key Results) "
    "system. Each engineer sets 3-5 objectives aligned with team goals. OKR "
    "completion rates are discussed during performance reviews but are not the "
    "sole basis for ratings.",

    "Meridian's data security policy requires all employees to complete annual "
    "cybersecurity training. Sensitive data must be encrypted at rest and in "
    "transit. Violations can result in disciplinary action up to termination.",

    "The company's promotion criteria include at least 18 months in the current "
    "role, consistently Strong Performer or above ratings, and a demonstrated "
    "impact beyond the immediate team. A promotion committee reviews all "
    "nominations quarterly.",

    "Meridian Technologies maintains a 401(k) retirement plan with a company "
    "match of up to 4% of the employee's salary. Vesting occurs over a 3-year "
    "schedule. Financial planning consultations are available at no cost.",

    "The onboarding process at Meridian spans two weeks and includes IT setup, "
    "team introductions, compliance training, and a buddy system pairing new "
    "hires with experienced employees. Feedback surveys are sent at 30, 60, "
    "and 90 days."
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
for i, doc in enumerate(KNOWLEDGE_BASE):
    print(f"  [{i:2d}] {doc[:70]}...")

## 2.4 — Build the Embedding Index

We encode every document with `bge-base-en-v1.5`, L2-normalise the vectors,
and insert them into a FAISS `IndexFlatIP` (inner-product = cosine similarity
after normalisation).

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

doc_embeddings = embed_model.encode(KNOWLEDGE_BASE, normalize_embeddings=True)
doc_embeddings = np.array(doc_embeddings, dtype="float32")

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"FAISS index built: {index.ntotal} vectors of dim {dimension}")

## 2.5 — Retrieval Function

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve the top-k most relevant documents for a query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")
    scores, indices = index.search(q_emb, top_k)
    return [KNOWLEDGE_BASE[i] for i in indices[0]]


# Quick sanity check
test_docs = retrieve("What is the remote work policy?")
for i, doc in enumerate(test_docs):
    print(f"[{i+1}] {doc[:90]}...")

## 2.6 — Load the LLM (Qwen3-8B, 4-bit)

We load `Qwen/Qwen3-8B` with NF4 quantisation via `bitsandbytes`. This model
will serve double duty: first as the **generator** producing RAG answers, then
as the **judge** evaluating those answers. We discuss the implications of this
dual role in §4.1.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

print(f"Model loaded: {model.config._name_or_path}")
print(f"Device map: {model.hf_device_map}")

## 3.1 — RAG Generation Function

The generation prompt instructs the model to answer *only* from the provided
context. We use greedy decoding (`do_sample=False`) for reproducibility.

In [ ]:
def generate_rag_answer(query: str, contexts: list[str]) -> str:
    """Generate an answer grounded in the retrieved contexts."""
    context_block = "\n".join(f"- {c}" for c in contexts)
    prompt = (
        "You are a helpful assistant. Answer the question based ONLY on the "
        "provided context. If the context does not contain enough information, "
        "say so explicitly.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 3.2 — Define Evaluation Rubrics

The rubrics are the heart of the Grouse-style evaluation. Each dimension has a
prose description and a 0–5 scoring scale where every level has *observable,
unambiguous* criteria. Well-written rubrics reduce judge variance and make scores
meaningful across evaluators.

> **Design principle:** Each rubric level should describe what the evaluator
> can *observe*, not what they must *infer*. “Several claims go beyond the
> context” is observable; “the model hallucinated” is interpretive.

In [ ]:
RUBRICS = {
    "groundedness": {
        "description": (
            "Is every claim in the answer supported by the retrieved context?"
        ),
        "scale": {
            5: "Every claim is directly supported by the context with no extrapolation.",
            4: "Almost all claims are supported; minor inferences are reasonable.",
            3: "Most claims are supported, but some lack clear evidence in context.",
            2: "Several claims go beyond what the context supports.",
            1: "Most claims are unsupported or fabricated.",
            0: "The answer is entirely disconnected from the context."
        }
    },
    "completeness": {
        "description": (
            "Does the answer cover all relevant information from the context?"
        ),
        "scale": {
            5: "Covers all relevant information from the context comprehensively.",
            4: "Covers most relevant information with minor omissions.",
            3: "Covers the main points but misses some relevant details.",
            2: "Significant relevant information from the context is missing.",
            1: "Only touches on a small fraction of relevant context.",
            0: "Fails to use any relevant information from the context."
        }
    },
    "relevance": {
        "description": (
            "Does the answer focus on what the user actually asked?"
        ),
        "scale": {
            5: "Directly and precisely answers the question asked.",
            4: "Answers the question with minor tangential content.",
            3: "Partially answers the question but includes off-topic material.",
            2: "Loosely related to the question but misses the core intent.",
            1: "Mostly off-topic with only incidental relevance.",
            0: "Completely unrelated to the question."
        }
    },
    "coherence": {
        "description": (
            "Is the answer well-structured, clear, and logically organised?"
        ),
        "scale": {
            5: "Perfectly structured, clear, and logical throughout.",
            4: "Well-structured with minor clarity issues.",
            3: "Generally clear but could be better organised.",
            2: "Somewhat disorganised or unclear in places.",
            1: "Poorly structured and difficult to follow.",
            0: "Incoherent or incomprehensible."
        }
    }
}

for dim, rubric in RUBRICS.items():
    print(f"{dim.upper()}: {rubric[\"description\"]}")

## 3.3 — Build the Judge Prompt Template

The judge prompt is carefully structured to minimise ambiguity. It presents the
rubric, the context, the question, and the answer — then asks for a justification
*before* the score. This “chain-of-thought” ordering is important: it forces the
judge to reason through the rubric before committing to a number, reducing
arbitrary scoring.

In [ ]:
def build_judge_prompt(
    query: str,
    contexts: list[str],
    answer: str,
    dimension: str
) -> str:
    """Construct the evaluation prompt for a single dimension."""
    rubric = RUBRICS[dimension]
    scale_text = "\n".join(
        f"  {score}: {desc}"
        for score, desc in sorted(rubric["scale"].items(), reverse=True)
    )
    context_text = "\n".join(
        f"[{i+1}] {c}" for i, c in enumerate(contexts)
    )

    return (
        "You are an impartial judge evaluating a RAG system's answer.\n\n"
        f"**Evaluation Dimension:** {dimension.upper()}\n"
        f"**Criteria:** {rubric[\"description\"]}\n\n"
        f"**Scoring Rubric:**\n{scale_text}\n\n"
        f"**Retrieved Context:**\n{context_text}\n\n"
        f"**User Question:** {query}\n\n"
        f"**System Answer:** {answer}\n\n"
        "**Instructions:**\n"
        f"1. Analyse the answer against the scoring rubric for {dimension}.\n"
        "2. Provide a brief justification (2-3 sentences).\n"
        "3. Assign a score from 0 to 5.\n\n"
        "Respond in this exact format:\n"
        "Justification: <your reasoning>\n"
        "Score: <integer 0-5>"
    )

## 3.4 — Judge Execution and Score Parsing

The judge function calls the LLM, then extracts the score using a regex parser
with fallback heuristics. Robust parsing is critical — the judge sometimes
embeds the score in slightly different formats.

In [ ]:
def parse_judge_output(text: str) -> tuple[int, str]:
    """Extract score and justification from judge output."""
    # Try structured format first
    score_match = re.search(r"Score:\s*(\d)", text)
    just_match = re.search(
        r"Justification:\s*(.+?)(?=Score:|$)", text, re.DOTALL
    )

    justification = just_match.group(1).strip() if just_match else text.strip()

    if score_match:
        return int(score_match.group(1)), justification

    # Fallback: find any digit 0-5 near the end of the text
    fallback = re.findall(r"\b([0-5])\b", text[-50:])
    if fallback:
        return int(fallback[-1]), justification

    return -1, justification  # Parsing failed


def judge_answer(
    query: str,
    contexts: list[str],
    answer: str,
    dimension: str
) -> dict:
    """Use the LLM to judge an answer on a specific dimension."""
    prompt = build_judge_prompt(query, contexts, answer, dimension)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(generated, skip_special_tokens=True).strip()
    score, justification = parse_judge_output(raw_output)

    return {
        "dimension": dimension,
        "score": score,
        "justification": justification,
        "raw_output": raw_output
    }

## 3.5 — Full Evaluation Function

The evaluation function runs the judge across all four dimensions and computes
an overall average. We also track any parsing failures so they can be inspected.

In [ ]:
def evaluate_rag_response(
    query: str,
    contexts: list[str],
    answer: str
) -> dict:
    """Evaluate a RAG response across all Grouse dimensions."""
    results = {}
    valid_scores = []

    for dimension in RUBRICS:
        result = judge_answer(query, contexts, answer, dimension)
        results[dimension] = result
        if result["score"] >= 0:
            valid_scores.append(result["score"])

    results["overall"] = (
        float(np.mean(valid_scores)) if valid_scores else -1.0
    )
    results["parse_failures"] = sum(
        1 for r in results.values()
        if isinstance(r, dict) and r.get("score", 0) < 0
    )
    return results

## 3.6 — Run the RAG Pipeline on Test Queries

We define a set of test queries spanning different topics in the knowledge base.
For each query we retrieve context, generate an answer, and store the triplet
for evaluation.

In [ ]:
test_queries = [
    "What is the company's remote work policy?",
    "How does the performance review process work?",
    "What benefits does the company offer for professional development?",
    "What health insurance options are available?",
    "How much PTO do employees receive?",
    "What are the requirements for promotion?",
    "What retirement benefits does the company provide?"
]

rag_results = []
for query in test_queries:
    contexts = retrieve(query)
    answer = generate_rag_answer(query, contexts)
    rag_results.append({
        "query": query,
        "contexts": contexts,
        "answer": answer
    })
    print(f"Q: {query}")
    print(f"A: {answer[:200]}...\n")

## 3.7 — Run Grouse-Style Evaluation

Now we run the judge on every RAG result. Each answer is evaluated across all
four dimensions, producing a score and justification for each.

In [ ]:
evaluation_results = []

for r in rag_results:
    eval_result = evaluate_rag_response(
        r["query"], r["contexts"], r["answer"]
    )
    evaluation_results.append(eval_result)

    print(f"Query: {r['query'][:60]}")
    for dim in RUBRICS:
        res = eval_result[dim]
        justification_preview = res["justification"][:80]
        print(f"  {dim:15s}: {res['score']}/5 — {justification_preview}...")
    print(f"  {'overall':15s}: {eval_result['overall']:.1f}/5")
    print()

## 4.1 — The Circular Evaluation Problem

In this notebook, we use Qwen3-8B as *both* the generator and the judge. This
creates a fundamental methodological concern: **the model is grading its own
homework.**

Research on LLM-as-judge has consistently documented **self-preference bias** —
the tendency of a model to rate its own outputs more favourably than outputs from
other models. When GPT-4 judges GPT-4 outputs, it assigns higher scores than
when judging Claude outputs of comparable quality, and vice versa. The same bias
applies here: Qwen3 judging Qwen3 outputs is likely to produce inflated scores.

Beyond raw score inflation, circular evaluation creates **blind spots**. The
model cannot catch errors that stem from its own systematic misunderstandings.
If it consistently misinterprets a type of query, it will also fail to recognise
that misinterpretation when judging.

### Mitigation Strategies

1. **Use a different (ideally stronger) model as judge.** This is the gold standard.
   In production, use GPT-4, Claude, or another frontier model as the judge while
   using a smaller model for generation.
2. **Structured rubrics constrain the judge.** Even a biased judge is more
   reliable when forced to evaluate against specific, observable criteria rather
   than open-ended quality assessment.
3. **Multiple judge prompts.** Evaluate the same answer with different prompt
   phrasings and average the scores to reduce prompt-specific biases.
4. **Calibrate against human annotations.** Score a small sample (20–50 cases)
   by hand, then compute the correlation between human and LLM scores. Adjust
   the rubric or add a correction factor.
5. **Focus on relative rankings, not absolute scores.** Even a biased judge
   can reliably rank answers from best to worst, even if absolute scores are
   inflated.

> **For this educational notebook,** using the same model demonstrates the
> concept without requiring API keys or a second model. In production, always
> separate the generator and judge models.

## 4.2 — Comparing Grouse-Style vs DeepEval Approaches

The DeepEval notebook in this repository (`evaluation_deep_eval.ipynb`) demonstrates
automated metric-based evaluation. How does it compare to the LLM-as-judge approach
we have built here?

| Aspect | DeepEval (Metrics) | Grouse (LLM Judge) |
|---|---|---|
| **Output** | Float score (0–1) | Score + justification |
| **Customisation** | Fixed metric definitions | Custom rubrics in natural language |
| **Interpretability** | Moderate — score only | High — explains reasoning |
| **Failure Diagnosis** | Indirect (score drops) | Direct (judge explains the failure) |
| **Best For** | CI/CD regression testing | Detailed debugging and analysis |
| **Speed** | Fast (decomposition-based) | Slower (full LLM inference per dimension) |
| **Reference Needed** | Sometimes | No |

Neither approach is strictly superior. DeepEval excels in automated pipelines where
you need fast, deterministic pass/fail signals. Grouse-style evaluation excels when
you need to *understand* why a system is failing and *how* to fix it.

## 4.3 — When to Use Each Approach

The best evaluation strategy uses **both** approaches at different stages of
the development lifecycle:

- **Continuous integration:** Run automated metrics (DeepEval-style) on every
  commit. They are fast, cheap, and catch regressions immediately.
- **Weekly deep dives:** Run LLM-judge evaluation (Grouse-style) on a sample
  of queries. Use the justifications to identify systematic issues.
- **Metric drops:** When automated metrics decline, trigger a Grouse-style
  evaluation to diagnose *why* the decline happened.
- **New domains:** When deploying RAG to a new domain, use Grouse-style
  evaluation to calibrate expectations before setting metric thresholds.

```
Practical Workflow:

  Commit → DeepEval metrics (fast) → Pass? → Ship
                                        ↓
                                      Fail?
                                        ↓
                               Grouse evaluation (detailed)
                                        ↓
                               Diagnose + Fix + Commit
```

## 4.4 — Building Custom Rubrics for Domain-Specific RAG

One of the most powerful features of the LLM-as-judge paradigm is the ability
to add domain-specific evaluation dimensions simply by writing new rubrics.
Below are two examples for specialised RAG applications.

In [ ]:
# Example: Medical RAG safety rubric
medical_rubric = {
    "safety": {
        "description": (
            "Does the answer avoid potentially harmful medical advice "
            "and appropriately recommend professional consultation?"
        ),
        "scale": {
            5: "Appropriately caveats all medical information and "
               "recommends professional consultation where needed.",
            4: "Generally safe with minor omissions in caveating.",
            3: "Provides medical info without consistent safety disclaimers.",
            2: "Makes specific recommendations without professional caveats.",
            1: "Provides potentially misleading medical guidance.",
            0: "Gives dangerous or contraindicated medical advice."
        }
    }
}

# Example: Legal RAG citation accuracy rubric
legal_rubric = {
    "citation_accuracy": {
        "description": (
            "Does the answer correctly cite specific laws, regulations, "
            "or legal precedents from the retrieved context?"
        ),
        "scale": {
            5: "All legal citations are accurate and properly attributed.",
            4: "Citations are mostly accurate with minor formatting issues.",
            3: "Some citations are correct but others are vague or incomplete.",
            2: "Multiple citation errors or attributions to wrong sources.",
            1: "Most citations are incorrect or fabricated.",
            0: "No citations provided or all are entirely wrong."
        }
    }
}

# Adding custom rubrics to the evaluation is straightforward:
extended_rubrics = {**RUBRICS, **medical_rubric}
print("Extended rubric dimensions:", list(extended_rubrics.keys()))

## 4.5 — Aggregating Results Across Test Sets

For a holistic view, we aggregate scores across all test queries and display
the results as a summary table. This reveals whether weaknesses are systemic
(low average across all queries) or isolated (one query pulling down the mean).

In [ ]:
# Build a summary table of average scores per dimension
print(f"{'Dimension':<15s} {'Mean':>6s} {'Min':>5s} {'Max':>5s}")
print("-" * 35)

dimension_scores = {dim: [] for dim in RUBRICS}

for eval_result in evaluation_results:
    for dim in RUBRICS:
        score = eval_result[dim]["score"]
        if score >= 0:
            dimension_scores[dim].append(score)

for dim, scores in dimension_scores.items():
    if scores:
        print(
            f"{dim:<15s} {np.mean(scores):>6.2f} "
            f"{min(scores):>5d} {max(scores):>5d}"
        )

overall_scores = [
    r["overall"] for r in evaluation_results if r["overall"] >= 0
]
print("-" * 35)
print(f"{'OVERALL':<15s} {np.mean(overall_scores):>6.2f}")

## 4.6 — Per-Query Detail View

For deeper debugging, inspect the full justification for each query and dimension.

In [ ]:
for i, (r, e) in enumerate(zip(rag_results, evaluation_results)):
    print(f"=== Query {i+1}: {r['query']} ===")
    print(f"Answer: {r['answer'][:150]}...")
    print()
    for dim in RUBRICS:
        res = e[dim]
        print(f"  [{dim.upper()}] Score: {res['score']}/5")
        print(f"    {res['justification'][:120]}")
    print(f"  Overall: {e['overall']:.1f}/5")
    print()

## 5.1 — Exercises

**Exercise 1 — Cross-Model Judging**

If you have access to a second LLM (e.g., via an API), set it up as the judge
while keeping Qwen3-8B as the generator. Compare the scores. Do you observe
systematic differences? Does the external judge produce lower scores (as
self-preference bias would predict)?

**Exercise 2 — Rubric Calibration**

Manually score 5 of the test cases yourself using the rubrics above. Then
compare your scores with the LLM judge's scores. Compute the Pearson or
Spearman correlation. Where do you and the model disagree most?

**Exercise 3 — Custom Domain Rubric**

Design a complete rubric (all four standard dimensions plus at least one custom
dimension) for a domain of your choice: medical Q&A, legal research, customer
support, or technical documentation. Test it by modifying the knowledge base
and running the full pipeline.

**Exercise 4 — Prompt Sensitivity**

Rewrite the judge prompt in three different styles (formal, concise, detailed)
and evaluate the same set of answers. How much do scores vary across prompt
styles? What does this tell you about judge robustness?

## 5.2 — Key Takeaways

1. **LLM-as-judge bridges the gap** between fast-but-shallow automated metrics
   and expensive-but-rich human evaluation.

2. **Rubric design is the most important step.** Well-written rubrics with
   observable, unambiguous criteria reduce judge variance and make scores
   comparable across evaluations.

3. **Four dimensions cover the core failure modes:** groundedness (hallucination),
   completeness (omission), relevance (off-topic), and coherence (readability).

4. **Circular evaluation is a real limitation.** Using the same model as generator
   and judge introduces self-preference bias. In production, always use a separate
   (ideally stronger) judge model.

5. **Justifications are the killer feature.** Unlike float scores, natural-language
   justifications tell you *what* to fix, not just *that* something is broken.

6. **Use both metrics and judges.** Automated metrics for CI/CD; LLM judges for
   deep debugging and continuous improvement.

7. **Custom rubrics unlock domain-specific evaluation** without writing new
   evaluation code — just write a new rubric in natural language.

## 5.3 — References

- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.*
  arXiv:2306.05685. The foundational paper on using LLMs as evaluators.
- Wang, P. et al. (2023). *Large Language Models are not Fair Evaluators.*
  arXiv:2305.17926. Documents position bias and self-preference bias in LLM judges.
- Es, S. et al. (2024). *RAGAS: Automated Evaluation of Retrieval Augmented Generation.*
  arXiv:2309.15217. Metric-based RAG evaluation framework.
- `evaluation_deep_eval.ipynb` — Companion notebook in this repository demonstrating
  automated metric-based evaluation with DeepEval.
- `simple_rag.ipynb` — Foundational RAG pipeline notebook in this repository.